In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install nltk langid malaya scikit-learn
# TensorFlow is usually pre-installed, if not: !pip install tensorflow

# Download NLTK data (punkt and stopwords)
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
!pip install langdetect

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from langdetect import detect, DetectorFactory
from sklearn.feature_extraction.text import TfidfVectorizer

# --- Language Detection Setup ---
# Option 1: Use langdetect (sometimes struggles with Malay/Manglish)
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0 # for reproducibility with langdetect

# Option 2: Use langid (often performs differently, sometimes better on short texts)
import langid
USE_LANGID = True # Set to False to revert to langdetect

# Initialize USE_MALAYA_STOPWORDS before the try block
USE_MALAYA_STOPWORDS = False
# Try importing Malaya
try:
    import malaya
    print("Malaya library imported successfully.")
    USE_MALAYA_STOPWORDS = True
except ImportError:
    print("Malaya library not found. Falling back to NLTK/custom stopwords for Malay.")
    USE_MALAYA_STOPWORDS = False
except Exception as e:
    print(f"Error importing Malaya: {e}. Falling back to NLTK/custom stopwords for Malay.")
    USE_MALAYA_STOPWORDS = False


# Ensure reproducibility for langdetect
DetectorFactory.seed = 0

# --- Configuration ---
# Path to raw data to process (CSV file)
input_csv_path = "malaysian_scams_tweets_selenium.csv"
# Output path for the preprocessed data
output_preprocessed_csv_path = "malaysian_scams_tweets_preprocessed.csv"

# --- Load Data ---
try:
    df = pd.read_csv(input_csv_path)
    print(f"Successfully loaded data from {input_csv_path}. Shape: {df.shape}")
    # Display first few rows to confirm
    print("\nOriginal Data Head:")
    print(df.head())

    # Ensure 'tweet_text' column exists. Change here if column name is different
    if 'tweet_text' not in df.columns:
        print("Error: 'tweet_text' column not found. Please check your CSV file and column name.")
        exit()

    # --- Remove duplicate tweet_text entries ---
    initial_rows = df.shape[0]
    df.drop_duplicates(subset=['tweet_text'], inplace=True)
    rows_after_dedup = df.shape[0]
    print(f"Removed {initial_rows - rows_after_dedup} duplicate 'tweet_text' entries.")
    print(f"New DataFrame shape after deduplication: {df.shape}")

    # Display first few rows to confirm
    print("\nOriginal Data Head (after deduplication):")
    print(df.head())

except FileNotFoundError:
    print(f"Error: The file '{input_csv_path}' was not found. Please check the path.")
    exit()
except Exception as e:
    print(f"An error occurred while loading the CSV: {e}")
    exit()

# --- Preprocessing Functions ---

def clean_text(text):
    """
    Removes URLs, mentions, emojis, hashtags, and special characters.
    """
    if not isinstance(text, str):
        return "" # Handle non-string inputs gracefully

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove mentions (@username)
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags (#hashtag)
    text = re.sub(r'#\w+', '', text)
    # Remove emojis (basic regex, might not catch all)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    # Remove special characters and numbers (keep letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.strip()

def lowercase_text(text):
    """
    Converts text to lowercase.
    """
    if not isinstance(text, str):
        return ""
    return text.lower()

def detect_language(text):
    """
    Detects language of the text. Returns 'en', 'ms', or 'other'.
    Uses langid if USE_LANGID is True, else langdetect.
    """
    if not isinstance(text, str) or len(text.strip()) < 3:
        return 'unknown'

    try:
        if USE_LANGID:
            lang, _ = langid.classify(text) # langid returns (language_code, confidence)
            if lang == 'en':
                return 'en'
            elif lang == 'ms':
                return 'ms'
            else:
                return 'other'
        else: # Fallback to langdetect
            lang = detect(text)
            if lang == 'en':
                return 'en'
            elif lang == 'ms':
                return 'ms'
            else:
                return 'other'
    except:
        return 'unknown' # Handle cases where language detection fails

def tokenize_text(text):
    """
    Tokenizes text into words.
    """
    if not isinstance(text, str):
        return []
    return word_tokenize(text)

def remove_stopwords_custom(tokens, lang='en'):
    """
    Removes stopwords from a list of tokens.
    Supports English and Malay (using Malaya if available, else NLTK/custom).
    """
    if not isinstance(tokens, list):
        return []

    stop_words = set()
    if lang == 'en':
        stop_words.update(stopwords.words('english'))
    elif lang == 'ms':
        # Access the USE_MALAYA_STOPWORDS variable from the outer scope
        global USE_MALAYA_STOPWORDS
        if USE_MALAYA_STOPWORDS:
            print("Using Malaya's Malay stopwords.")
            try:
                malaya_ms_stopwords = malaya.text.function.get_stopwords()
                stop_words.update(malaya_ms_stopwords)
            except AttributeError:
                print("Error: Could not access Malaya stopwords. Falling back to NLTK/custom.")
                USE_MALAYA_STOPWORDS = False # Update flag to prevent future attempts

        if not USE_MALAYA_STOPWORDS:
            print("Using NLTK/custom Malay stopwords.")
            try:
                stop_words.update(stopwords.words('malay'))
            except LookupError:
                # Fallback if 'malay' stopwords are not in NLTK
                print("Warning: NLTK Malay stopwords not found. Using a basic custom list.")
                custom_malay_stopwords = ['yang', 'dan', 'ini', 'itu', 'untuk', 'dengan', 'saya', 'tidak', 'adalah', 'di', 'ke', 'dari', 'akan', 'juga', 'tetapi', 'atau', 'pada', 'sudah', 'telah', 'masih', 'lagi', 'bukan', 'ia', 'mereka', 'kami', 'kita']
                stop_words.update(custom_malay_stopwords)
            except Exception as e:
                print(f"An unexpected error occurred with NLTK Malay stopwords: {e}. Using a basic custom list.")
                custom_malay_stopwords = ['yang', 'dan', 'ini', 'itu', 'untuk', 'dengan', 'saya', 'tidak', 'adalah', 'di', 'ke', 'dari', 'akan', 'juga', 'tetapi', 'atau', 'pada', 'sudah', 'telah', 'masih', 'lagi', 'bukan', 'ia', 'mereka', 'kami', 'kita']
                stop_words.update(custom_malay_stopwords)


    return [word for word in tokens if word.lower() not in stop_words] # Ensure lowercase check for stopwords

# --- Apply Preprocessing Steps ---

print("\nStarting preprocessing...")

# 1. Lowercasing
df['processed_text'] = df['tweet_text'].apply(lowercase_text)
print("Step 1: Lowercasing complete.")

# 2. Noise Removal (URLs, mentions, emojis, hashtags, special characters)
df['processed_text'] = df['processed_text'].apply(clean_text)
print("Step 2: Noise removal complete.")

# 3. Language Detection and Filtering (Optional for filtering, but good for analysis)
df['detected_language'] = df['processed_text'].apply(detect_language)
print("Step 3: Language detection complete.")
print(f"Language distribution:\n{df['detected_language'].value_counts()}")

# 4. Tokenization
df['tokens'] = df['processed_text'].apply(tokenize_text)
print("Step 4: Tokenization complete.")

# 5. Stop-word Removal (Apply based on detected language)
# This loop applies language-specific stopword removal using the 'detected_language' column.
df['tokens_no_stopwords'] = df.apply(
    lambda row: remove_stopwords_custom(row['tokens'], lang=row['detected_language']),
    axis=1
)
print("Step 5: Stop-word removal complete.")

# Rejoin tokens into a string for traditional ML models (TF-IDF)
df['final_text_for_traditional_ml'] = df['tokens_no_stopwords'].apply(lambda tokens: ' '.join(tokens))
print("Final text prepared for traditional ML models.")

# --- Label Encoding (Placeholder) ---
print("\n--- Label Encoding ---")
print("Important: Need a 'label' column (e.g., 0 for Legit, 1 for Scam) in the DataFrame for model training.")
print("Need to manually label a subset of the collected data.")

# --- Vectorization (for Traditional ML: SVM + TF-IDF) ---
print("\n--- Vectorization for Traditional ML (TF-IDF) ---")
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['final_text_for_traditional_ml'])

print(f"TF-IDF Vectorization complete. Shape of TF-IDF matrix: {tfidf_matrix.shape}")
print("This matrix (tfidf_matrix) is ready to be used with SVM (along with the 'label' column).")

# --- Save Preprocessed Data ---
df.to_csv(output_preprocessed_csv_path, index=False, encoding='utf-8')
print(f"\nPreprocessed data saved to {output_preprocessed_csv_path}")

print("\nPreprocessing script finished.")
print("Remember to manually add a 'label' column to the CSV or DataFrame for training the models.")

In [ ]:
import nltk
nltk.download('stopwords', quiet=True) # Ensure the main stopwords are downloaded
try:
    nltk.data.find('corpora/stopwords/malay')
    print("NLTK Malay stopwords already downloaded.")
except LookupError:
    print("Attempting to download NLTK Malay stopwords...")
    # Download the specific 'stopwords' corpus, which should include 'malay' if available
    nltk.download('stopwords')
    try:
         nltk.data.find('corpora/stopwords/malay')
         print("NLTK Malay stopwords downloaded successfully.")
    except LookupError:
         print("Failed to download NLTK Malay stopwords. It might not be available or check the NLTK data path.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
# Telegram dataset
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from langdetect import detect, DetectorFactory
import langid
from sklearn.feature_extraction.text import TfidfVectorizer

# --- Setup ---
DetectorFactory.seed = 0
USE_LANGID = True  # toggle between langid and langdetect
USE_MALAYA_STOPWORDS = False

# Try importing Malaya for Malay stopwords
try:
    import malaya
    USE_MALAYA_STOPWORDS = True
    print("Malaya library imported successfully.")
except ImportError:
    print("Malaya not found. Falling back to NLTK/custom stopwords for Malay.")
except Exception as e:
    print(f"Error importing Malaya: {e}. Using fallback.")

# --- Configuration ---
input_csv_path = "scam_dataset_raw.csv"
output_preprocessed_csv_path = "scam_dataset_preprocessed.csv"

# --- Load Data ---
try:
    df = pd.read_csv(input_csv_path)
    print(f"Loaded data from {input_csv_path}. Shape: {df.shape}")
    print(df.head())

    # Ensure 'text' column exists
    if 'text' not in df.columns:
        raise ValueError("Error: 'text' column not found in dataset.")

    # Remove duplicates
    initial_rows = df.shape[0]
    df.drop_duplicates(subset=['text'], inplace=True)
    print(f"Removed {initial_rows - df.shape[0]} duplicate rows.")

except FileNotFoundError:
    print(f"Error: File '{input_csv_path}' not found.")
    raise
except Exception as e:
    print(f"Error while loading CSV: {e}")
    raise

# --- Preprocessing Functions ---
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # URLs
    text = re.sub(r'@\w+', '', text)  # mentions
    text = re.sub(r'#\w+', '', text)  # hashtags
    emoji_pattern = re.compile("["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # keep only letters
    return text.strip()

def lowercase_text(text):
    return text.lower() if isinstance(text, str) else ""

def detect_language(text):
    if not isinstance(text, str) or len(text.strip()) < 3:
        return 'unknown'
    try:
        if USE_LANGID:
            lang, _ = langid.classify(text)
        else:
            lang = detect(text)
        return lang if lang in ['en', 'ms'] else 'other'
    except:
        return 'unknown'

def tokenize_text(text):
    return word_tokenize(text) if isinstance(text, str) else []

def remove_stopwords_custom(tokens, lang='en'):
    if not isinstance(tokens, list):
        return []
    stop_words = set()
    if lang == 'en':
        stop_words.update(stopwords.words('english'))
    elif lang == 'ms':
        global USE_MALAYA_STOPWORDS
        if USE_MALAYA_STOPWORDS:
            try:
                malaya_ms_stopwords = malaya.text.function.get_stopwords()
                stop_words.update(malaya_ms_stopwords)
            except:
                USE_MALAYA_STOPWORDS = False
        if not USE_MALAYA_STOPWORDS:
            try:
                stop_words.update(stopwords.words('malay'))
            except:
                stop_words.update(['yang', 'dan', 'ini', 'itu', 'untuk', 'dengan',
                                   'saya', 'tidak', 'adalah', 'di', 'ke', 'dari',
                                   'akan', 'juga', 'tetapi', 'atau', 'pada',
                                   'sudah', 'telah', 'masih', 'lagi', 'bukan',
                                   'ia', 'mereka', 'kami', 'kita'])
    return [word for word in tokens if word.lower() not in stop_words]

# --- Apply Preprocessing ---
print("\nStarting preprocessing...")

df['processed_text'] = df['text'].apply(lowercase_text)
df['processed_text'] = df['processed_text'].apply(clean_text)
df['detected_language'] = df['processed_text'].apply(detect_language)
df['tokens'] = df['processed_text'].apply(tokenize_text)
df['tokens_no_stopwords'] = df.apply(
    lambda row: remove_stopwords_custom(row['tokens'], lang=row['detected_language']),
    axis=1
)
df['final_text_for_traditional_ml'] = df['tokens_no_stopwords'].apply(lambda tokens: ' '.join(tokens))

print("Preprocessing complete. Sample:")
print(df[['text', 'processed_text', 'final_text_for_traditional_ml', 'label']].head())

# --- Vectorization for SVM ---
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['final_text_for_traditional_ml'])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# --- Preparation for mBERT ---
print("\nFor mBERT, use the 'processed_text' column directly with Hugging Face tokenizer.")

# --- Save Preprocessed Data ---
df.to_csv(output_preprocessed_csv_path, index=False, encoding='utf-8')
print(f"Saved preprocessed data to {output_preprocessed_csv_path}")
